# Lab 10 · LangChain: an agent loop with enforceable limits
## Goal
Use `create_agent`, real tool messages, custom tool middleware, and model-call limits. Compare a high-level agent factory with the explicit graph from Lab 9 and the OpenAI Agents SDK in Lab 7. Prerequisites: Labs 5–7; textbook chapters 3, 5, 11, and 12.

Estimated 110 minutes: 15 architecture, 20 trace, 40 implementation, 20 failure injection, 15 comparison. You will test success, rejected arguments, and a model that never stops asking for tools. The deterministic model below scripts responses; it is not a language model and cannot establish inference quality.

## Setup
Use the separate `labs/frameworks/requirements.txt` environment described in this track's README. Do not upgrade the earlier lab environment in place. Each notebook is self-contained after installation. Python 3.12 is the tested interpreter family.

Default execution uses invented equipment-service fixtures and real framework runtimes, without network inference. These checks establish application behavior, not model accuracy. Student tasks are disabled until you implement them and set `RUN_EXERCISES=True`. Instructor copies enable them. Optional live cells require explicit `RUN_LIVE=True`, `GEMINI_API_KEY`, and `GEMINI_MODEL`. Choose a model available to your account; API use consumes quota. Keep credentials out of notebook cells and saved output.

In [1]:
import os
from importlib.metadata import version
RUN_EXERCISES = True
RUN_LIVE = False

def live_config():
    key, model = os.getenv("GEMINI_API_KEY"), os.getenv("GEMINI_MODEL")
    if not key or not model:
        raise RuntimeError("Set GEMINI_API_KEY and GEMINI_MODEL before enabling live calls")
    return key, model

print({p: version(p) for p in ("langgraph", "langchain", "pydantic-ai-slim")})

{'langgraph': '1.2.14', 'langchain': '1.4.3', 'pydantic-ai-slim': '2.54.0'}


## Steps
### 1. Implement a narrow tool and a scripted model
The tool returns an invented inventory count. A model call requests the tool; a second call incorporates its result. `bind_tools` is a testing adapter here: it returns the scripted model unchanged. A provider integration must actually bind and transmit the schema. Inspect the returned message sequence, not only the final string.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, wrap_tool_call
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult

INVENTORY = {"camera":2, "tripod":0}

def stock(item: str) -> str:
    """Return the available count for a known equipment item."""
    if item not in INVENTORY:
        raise ValueError("unknown equipment")
    return str(INVENTORY[item])

class ScriptedInventoryModel(BaseChatModel):
    item: str = "camera"
    repeat: bool = False
    @property
    def _llm_type(self): return "course-scripted-inventory"
    def bind_tools(self, tools, **kwargs): return self
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
        if tool_messages and not self.repeat:
            message = AIMessage(content="Inventory response: " + str(tool_messages[-1].content))
        else:
            message = AIMessage(content="", tool_calls=[{"name":"stock", "args":{"item":self.item}, "id":f"stock-{len(tool_messages)}", "type":"tool_call"}])
        return ChatResult(generations=[ChatGeneration(message=message)])

@wrap_tool_call
def safe_tool_errors(request, handler):
    try:
        return handler(request)
    except ValueError:
        return ToolMessage(content="invalid_equipment", tool_call_id=request.tool_call["id"], status="error")

agent = create_agent(ScriptedInventoryModel(), tools=[stock], middleware=[safe_tool_errors, ModelCallLimitMiddleware(run_limit=3, exit_behavior="error")])
trace = agent.invoke({"messages":[{"role":"user", "content":"How many cameras?"}]})
assert trace["messages"][-1].content == "Inventory response: 2"
print([(m.type, m.content) for m in trace["messages"]])

[('human', 'How many cameras?'), ('ai', ''), ('tool', '2'), ('ai', 'Inventory response: 2')]


### 2. Separate protocol correctness from answer quality
Check that the ToolMessage refers to the original tool-call ID. The error middleware exposes a stable error code; it does not silently invent a count. Returning an error message permits recovery, but the scripted model merely repeats that code. Application success still needs an independent criterion.

A model-call cap bounds loop iterations involving the model. It is not a token or currency budget, and parallel tool calls can require an additional tool-call limit. A run cap resets on a new invocation; a thread cap requires persisted thread state.

In [3]:
unknown = create_agent(ScriptedInventoryModel(item="spaceship"), tools=[stock], middleware=[safe_tool_errors])
failed = unknown.invoke({"messages":[{"role":"user","content":"Check equipment"}]})
tool_reply = next(m for m in failed["messages"] if isinstance(m, ToolMessage))
assert tool_reply.status == "error" and tool_reply.content == "invalid_equipment"
assert tool_reply.tool_call_id == "stock-0"
print("Unknown equipment was rejected; no count was fabricated.")

Unknown equipment was rejected; no count was fabricated.


### 3. TODO: construct a bounded agent (35 points)
Implement `bounded_agent(model, limit)` using `create_agent`, `stock`, `safe_tool_errors`, and `ModelCallLimitMiddleware`. Accept only integers 1–5 (reject booleans) and use `exit_behavior="error"`. Invalid limits must raise `ValueError` before building the agent.

In [4]:
def bounded_agent(model, limit):
    if type(limit) is not int or not 1 <= limit <= 5:
        raise ValueError("limit must be an integer from 1 to 5")
    return create_agent(model, tools=[stock], middleware=[safe_tool_errors, ModelCallLimitMiddleware(run_limit=limit, exit_behavior="error")])

In [5]:
if RUN_EXERCISES:
    from langchain.agents.middleware.model_call_limit import ModelCallLimitExceededError
    request = {"messages":[{"role":"user","content":"Check camera stock"}]}
    assert bounded_agent(ScriptedInventoryModel(), 2).invoke(request)["messages"][-1].content.endswith("2")
    for limit in (1,2,5):
        try: bounded_agent(ScriptedInventoryModel(repeat=True), limit).invoke(request)
        except ModelCallLimitExceededError: pass
        else: raise AssertionError("runaway model escaped budget")
    for bad in (True,0,6,-1,1.5,None,"2"):
        try: bounded_agent(ScriptedInventoryModel(), bad)
        except ValueError: pass
        else: raise AssertionError("invalid limit accepted")

### 4. TODO: audit the message protocol (35 points)
Implement `audit_trace(messages)` returning `{"calls": n, "results": n}` for a complete trace. Across AI messages collect tool-call IDs; reject duplicate IDs, tool results without a preceding call, duplicate results, and a trace ending with unresolved calls. Raise `ValueError` for those protocol errors. Empty input is a valid zero-call trace. Assume elements are LangChain message objects.

In [6]:
def audit_trace(messages):
    seen, resolved = set(), set()
    for message in messages:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                if call["id"] in seen:
                    raise ValueError("duplicate call")
                seen.add(call["id"])
        elif isinstance(message, ToolMessage):
            key = message.tool_call_id
            if key not in seen or key in resolved:
                raise ValueError("orphan or duplicate result")
            resolved.add(key)
    if seen != resolved:
        raise ValueError("unresolved calls")
    return {"calls":len(seen), "results":len(resolved)}

In [7]:
if RUN_EXERCISES:
    assert audit_trace([]) == {"calls":0,"results":0}
    assert audit_trace(trace["messages"]) == {"calls":1,"results":1}
    call = next(m for m in trace["messages"] if isinstance(m,AIMessage) and m.tool_calls)
    reply = next(m for m in trace["messages"] if isinstance(m,ToolMessage))
    for bad in ([reply], [call], [call,call], [call,reply,reply]):
        try: audit_trace(bad)
        except ValueError: pass
        else: raise AssertionError("bad protocol accepted")
    print("Student agent and protocol checks passed")

Student agent and protocol checks passed


### 5. Optional Gemini integration
Replace only the model adapter and retain the same tool and call-limit middleware. This uses the provider SDK through LangChain; the live answer may differ and may omit a tool call. Inspect that behavior rather than asserting an exact text match. This cell is off by default and has not been validated against a live account.

In [8]:
if RUN_LIVE:
    from langchain_google_genai import ChatGoogleGenerativeAI
    key, model_name = live_config()
    live_model = ChatGoogleGenerativeAI(model=model_name, google_api_key=key, temperature=0, max_retries=0)
    live_agent = create_agent(live_model, tools=[stock], middleware=[safe_tool_errors, ModelCallLimitMiddleware(run_limit=3, exit_behavior="error")])
    live_result = live_agent.invoke({"messages":[{"role":"user", "content":"Use stock to find the number of cameras currently available."}]})
    print(live_result["messages"][-1].content)
else:
    print("Live Gemini request skipped.")

Live Gemini request skipped.


## Checks and submission
Submit the notebook plus a comparison with Lab 7: where are tools declared, who runs the loop, how is termination bounded, and what makes a run successful? Add a multi-tool-call message to the audit tests and explain why a call-limit exception is a recorded failure, not a successful answer. Do not compare framework quality using the scripted model's success rate.

Rubric: bounded agent 35; protocol audit 35; added failure tests 15; architecture comparison 15. Estimated timing is a teaching plan, not a measured completion statistic.

## Next Steps
Evaluate live models on the same held-out cases and budgets before selecting an implementation. Keep transport retries distinct from model reasoning turns. Extend the tool boundary to use request-scoped authorization if it ever performs writes.

References checked 2026-10-06: [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents), [middleware and model-call limits](https://docs.langchain.com/oss/python/langchain/middleware/built-in), [Gemini integration](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai).